<a href="https://colab.research.google.com/github/toryor31oct/group15-Fun-Rai-Kwam-plod-Phai/blob/Tonyong/Final_fun_rai_kwam_plod_phai(4).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# โหลดไฟล์ข้อมูลการเคลม

In [5]:
import pandas as pd

# (มีข้อมูลลูกค้าและกรมธรรม์อยู่แล้ว)
df_claims = pd.read_csv("claims_system_300.csv")

# ตั้งค่าฟอร์แมตตัวเลข
pd.options.display.float_format = "{:,.2f}".format


# ตั้งคำถามเกี่ยวกับธุรกิจแล้วตอบด้วย pandas
คำถามที่ 1: ประกันแต่ละประเภทมี "ยอดรวมเงินจ่ายสุทธิ" และ "ยอดขอเบิกเฉลี่ย" เท่าใด

คำถามที่ 2: ภาพรวมสถานะการอนุมัติ (Approved vs Rejected) มีจำนวนกี่เคส และจ่ายเงินสุทธิรวมเท่าใด

คำถามที่ 3: เมื่อจำแนกตามประเภทประกัน มีจำนวนเคสที่ผ่านอนุมัติและถูกปฏิเสธอย่างไรบ้าง?

คำถามที่ 4: ผู้รับประโยชน์กลุ่มไหน (บิดา, มารดา, บุตร ฯลฯ) มีจำนวนเคสมากสุด พร้อมยอดจ่ายสูงสุดและค่าเฉลี่ย?

In [6]:
# คำถามที่ 1: ประกันแต่ละประเภทมี "ยอดรวมเงินจ่ายสุทธิ" และ "ยอดขอเบิกเฉลี่ย" เท่าใด

q1_df = (df_claims.groupby("insurance_type").agg(
        total_net_payout = ("net_payout", "sum"),
        avg_claim_amount = ("claim_amount", "mean"),).sort_values(by="total_net_payout", ascending=False))
display(q1_df)

,total_net_payout,avg_claim_amount
insurance_type,,
ประกันชีวิต,"137,857,500.00","1,071,699.13"
ประกันโรคร้ายแรง,"81,972,000.00","664,804.59"
ประกันสุขภาพ,"26,216,579.78","469,881.22"
ประกันอุบัติเหตุ,"19,886,779.63","325,320.82"


In [7]:
# คำถามที่ 2: ภาพรวมสถานะการอนุมัติ (Approved vs Rejected) มีจำนวนกี่เคส และจ่ายเงินสุทธิรวมเท่าใด?

q2_df = (df_claims.groupby("approval_status").agg(
         total_request = ("claim_id", "count"),
         total_payout = ("net_payout", "sum"),).sort_values(by="total_request", ascending=False))

display(q2_df)

,total_request,total_payout
approval_status,,
Approved,276,"265,932,859.41"
Rejected,24,0.00


In [8]:
# คำถามที่ 3: เมื่อจำแนกตามประเภทประกัน มีจำนวนเคสที่ผ่านอนุมัติและถูกปฏิเสธอย่างไรบ้าง?

# part1 จำนวนเคสที่ผ่านอนุมัติและถูกปฏิเสธ
q3_df = (df_claims.groupby(["insurance_type", "payment_status"]).agg(
         total_cases = ("claim_id", "count"),
         total_payout = ("net_payout", "sum"),).sort_values(by=["insurance_type", "payment_status"], ascending=False))
print("=== จำนวนเคสและยอดจ่ายรวม ( Completed = ผ่าน | Cancelled = ไม่ผ่าน ) ===")
display(q3_df)

# part2 เหตุผลที่ไม่ผ่าน พร้อมส่วนต่างเงินที่ยื่นเกินวงเงิน
df_rejected = df_claims[df_claims["payment_status"] == "Cancelled"].copy()
df_rejected["excess_amount"] = (df_rejected["claim_amount"] - df_rejected["coverage_limit"])

q3_detail_df = (df_rejected.groupby(["insurance_type", "claim_detail"]).agg(
                rejected_cases = ("claim_id", "count"),
                avg_claim_amount = ("claim_amount", "mean"),
                avg_coverage_limit = ("coverage_limit", "mean"),
                avg_axcess_amount = ("excess_amount", "mean"),).sort_values(by="rejected_cases", ascending=False))

print("\n=== เหตุผลที่ไม่ผ่าน และยอดเงินที่ยื่นเกินวงเงิน ==="
)
display(q3_detail_df)

=== จำนวนเคสและยอดจ่ายรวม ( Completed = ผ่าน | Cancelled = ไม่ผ่าน ) ===


total_cases   total_payout
insurance_type   payment_status                            
ประกันโรคร้ายแรง Paid                     70  81,972,000.00
ประกันอุบัติเหตุ Paid                     68  19,886,779.63
                 Cancelled                 7           0.00
ประกันสุขภาพ     Paid                     63  26,216,579.78
                 Cancelled                17           0.00
ประกันชีวิต      Paid                     75 137,857,500.00


=== เหตุผลที่ไม่ผ่าน และยอดเงินที่ยื่นเกินวงเงิน ===


rejected_cases  \
insurance_type   claim_detail                                                    
ประกันสุขภาพ     ผ่าตัดไส้ติ่งอักเสบฉุกเฉิน                                  6   
                 เข้ารับการรักษาผู้ป่วยใน (IPD) โรคไข้หวัดใหญ่               6   
                 เข้ารับการรักษาผู้ป่วยนอก (OPD) ลำไส้อักเสบ                 5   
ประกันอุบัติเหตุ เข้ารักษาฉุกเฉินจากลื่นล้มหกล้ม                             4   
                 กระดูกข้อเท้าแตกหักจากการเล่นกีฬา                           3   

                                                                avg_claim_amount  \
insurance_type   claim_detail                                                      
ประกันสุขภาพ     ผ่าตัดไส้ติ่งอักเสบฉุกเฉิน                           689,911.16   
                 เข้ารับการรักษาผู้ป่วยใน (IPD) โรคไข้หวัดใหญ่        712,611.01   
                 เข้ารับการรักษาผู้ป่วยนอก (OPD) ลำไส้อักเสบ          538,794.15   
ประกันอุบัติเหตุ เข้ารักษาฉุกเฉินจากลื่นล้มหกล้ม                      537,886.95   
                 กระดูกข้อเท้าแตกหักจากการเล่นกีฬา                    719,952.44   

                                                                avg_coverage_limit  \
insurance_type   claim_detail                                                        
ประกันสุขภาพ     ผ่าตัดไส้ติ่งอักเสบฉุกเฉิน                             675,000.00   
                 เข้ารับการรักษาผู้ป่วยใน (IPD) โรคไข้หวัดใหญ่          650,000.00   
                 เข้ารับการรักษาผู้ป่วยนอก (OPD) ลำไส้อักเสบ            500,000.00   
ประกันอุบัติเหตุ เข้ารักษาฉุกเฉินจากลื่นล้มหกล้ม                        512,500.00   
                 กระดูกข้อเท้าแตกหักจากการเล่นกีฬา                      683,333.33   

                                                                avg_axcess_amount  
insurance_type   claim_detail                                                      
ประกันสุขภาพ     ผ่าตัดไส้ติ่งอักเสบฉุกเฉิน                             14,911.16  
                 เข้ารับการรักษาผู้ป่วยใน (IPD) โรคไข้หวัดใหญ่          62,611.01  
                 เข้ารับการรักษาผู้ป่วยนอก (OPD) ลำไส้อักเสบ            38,794.15  
ประกันอุบัติเหตุ เข้ารักษาฉุกเฉินจากลื่นล้มหกล้ม                        25,386.96  
                 กระดูกข้อเท้าแตกหักจากการเล่นกีฬา                      36,619.11

In [9]:
# คำถามที่ 4: ผู้รับประโยชน์กลุ่มไหน (บิดา, มารดา, บุตร ฯลฯ) มีจำนวนเคสมากสุด พร้อมยอดจ่ายสูงสุดและค่าเฉลี่ย?

q4_df = (df_claims.groupby("beneficiary_relationship").agg(
         total_cases = ("claim_id", "count"),
         max_net_payout = ("net_payout", "max"),
         avg_net_payout = ("net_payout", "mean"),).sort_values(by="total_cases", ascending=False))

display(q4_df)

,total_cases,max_net_payout,avg_net_payout
beneficiary_relationship,,,
บุตร,60,"2,722,500.00","866,835.03"
พี่น้อง,51,"2,920,500.00","967,573.84"
สามี,51,"2,871,000.00","719,399.80"
ภรรยา,51,"2,970,000.00","977,530.77"
บิดา,45,"2,920,500.00","948,092.90"
มารดา,42,"2,920,500.00","842,115.52"
